In [0]:
# ============================================================
# PARAMÈTRES
# ============================================================
dbutils.widgets.text("catalog",        "banking")
dbutils.widgets.text("schema_bronze",  "bronze")
dbutils.widgets.text("schema_silver",  "silver")

catalog       = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")

# Tables
source_table     = f"{catalog}.{schema_bronze}.bronze_credit"
target_silver    = f"{catalog}.{schema_silver}.silver_credit"
monitoring_table = f"{catalog}.{schema_bronze}.execution_monitoring"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable


df = spark.table(source_table)
# ── Transformations
print('le nombres des lignes avant' ,df.count())

df=df.filter(F.col('customer_id').isNotNull())
df=df.filter(F.col('bureau_pull_date').isNotNull())
print('le nombres des lignes apres' ,df.count())

try:
    date_max = spark.sql(f"""SELECT COALESCE(MAX(bureau_pull_date), '1900-01-01')
        FROM {target_silver}""").collect()[0][0]
    print(f'Watermark: {date_max}')
except :
   
    date_max = '1900-01-01'

# Filter by watermark early
df = df.filter(F.col("bureau_pull_date") > date_max)

# Handle duplicates
window = Window.partitionBy('customer_id').orderBy(F.col('bureau_pull_date').desc())
df = df.withColumn('row_number', F.row_number().over(window))
df = df.filter(F.col('row_number') == 1).drop('row_number')

# Add silver timestamp
df = df.withColumn('silver_loaded_at', F.current_timestamp())

print(f'Rows after transformations: {df.count()}')

# Check if table exists
table_exists = spark.catalog.tableExists(target_silver)

# Write or merge
if not table_exists:
    df.write.format("delta").mode("overwrite").option("delta.feature.allowColumnDefaults", "supported").saveAsTable(target_silver)
    print(f"Table {target_silver} created successfully")
else:
    df.write.format("delta").mode("append").option("delta.feature.allowColumnDefaults", "supported").saveAsTable(target_silver)
    print("✅ Table silver_customers créée")
  